In [ ]:
# VS写的，融合策略：动量轮动+海龟交易+风险控制
from bigquant import bigtrader, dai
import math
import numpy as np
import pandas as pd

def initialize(context: bigtrader.IContext):
    """策略初始化"""
    # 设置交易成本
    context.set_commission(bigtrader.PerOrder(buy_cost=0.0002, sell_cost=0.0002, min_cost=5))
    
    # ETF池 - 结合两个策略的标的
    context.etf_pool = [
        '518880.SH',  # 黄金ETF
        '513100.SH',  # 纳指100ETF
        '159941.SZ',  # 纳指ETF
        '159915.SZ',  # 创业板100
        '510300.SH',  # 沪深300ETF
        '511090.SH',  # 国债ETF
        '513030.SH',  # 德国ETF
        '159329.SH',  # 沙特ETF
    ]
    
    # 动量参考天数
    context.m_days = 20
    
    # 目标持仓数量
    context.target_num = 3
    
    # 风险控制参数
    context.value_shrink = 0.85  # 总资金回撤阈值（15%）- 放宽阈值避免过早触发
    context.stop_loss_rate = 0.05  # 止损阈值（5%）
    context.drawback_rate = 0.08  # 回撤止损阈值（8%）
    context.rest_days = 20  # 回撤后休息天数
    
    # 状态变量
    context.max_net_value = context.capital_base  # 历史最高资产净值
    context.is_resting = False  # 是否处于休息状态
    context.rest_counter = 0   # 休息天数计数器
    
    # 海龟策略参数
    context.window_size = 20  # 计算20日高点低点
    
    context.logger.info("开始融合策略回测...")
    context.logger.info(f"ETF池: {context.etf_pool}")
    context.logger.info(f"动量天数: {context.m_days}")
    
    # 获取ETF历史价格数据
    etf_codes_str = "', '".join(context.etf_pool)
    sql_price = f"""
    SELECT 
        date,
        instrument,
        open,
        high,
        low,
        close,
        volume
    FROM cn_fund_bar1d 
    WHERE instrument IN ('{etf_codes_str}')
    ORDER BY date, instrument
    """
    
    df_price = dai.query(sql_price, filters={
        "date": [context.add_trading_days(context.start_date, -60), context.end_date]
    }).df()
    
    context.logger.info(f"价格数据查询完成: {len(df_price)} 条记录")
    context.logger.info(f"日期范围: {df_price['date'].min()} 到 {df_price['date'].max()}")
    
    if len(df_price) == 0:
        context.logger.warning("未找到ETF价格数据")
        context.data = pd.DataFrame(columns=['date', 'instrument', 'weight'])
        return

    def MOM(close_prices):
        """
        动量评分函数
        """
        if len(close_prices) < context.m_days:
            return None
            
        try:
            # 价格取对数
            y = np.log(close_prices)
            n = len(y)  
            x = np.arange(n)
            # 权重线性递增
            weights = np.linspace(1, 2, n)  
            # 加权线性回归
            slope, intercept = np.polyfit(x, y, 1, w=weights)
            # 年化收益率
            annualized_returns = math.pow(math.exp(slope), 250) - 1
            # 残差
            residuals = y - (slope * x + intercept)
            # 加权残差平方和
            weighted_residuals = weights * residuals**2
            # R平方
            r_squared = 1 - (np.sum(weighted_residuals) / np.sum(weights * (y - np.mean(y))**2))
            # 最终得分
            score = annualized_returns * r_squared
            return score
        except:
            return None

    def get_rank(etf_data, date, m_days):
        """
        基于年化收益和判定系数打分的动量因子轮动
        """
        score_list = []
        etf_list = []
        
        for etf in context.etf_pool:
            etf_hist = etf_data[etf_data['instrument'] == etf]
            etf_hist = etf_hist[etf_hist['date'] <= date].tail(m_days)
            
            if len(etf_hist) >= m_days:
                close_prices = etf_hist['close'].values
                score = MOM(close_prices)
                
                if score is not None:
                    score_list.append(score)
                    etf_list.append(etf)
        
        if len(score_list) == 0:
            return []
            
        df = pd.DataFrame(index=etf_list, data={'score': score_list})
        df = df.sort_values(by='score', ascending=False)
        
        # 安全区间，动量过高过低都不好
        df = df[(df['score'] > 0) & (df['score'] <= 5)]
        
        rank_list = list(df.index)
        
        return rank_list

    def get_turtle_signals(etf_data, date, window_size):
        """
        海龟交易策略信号
        """
        signals = {}
        
        for etf in context.etf_pool:
            etf_hist = etf_data[etf_data['instrument'] == etf]
            etf_hist = etf_hist[etf_hist['date'] <= date].tail(window_size)
            
            if len(etf_hist) >= window_size:
                # 计算20日最高价和最低价
                high_20 = etf_hist['high'].max()
                low_20 = etf_hist['low'].min()
                current_price = etf_hist['close'].iloc[-1]
                
                # 开仓信号：突破20日最高价
                if current_price > high_20:
                    signals[etf] = 'long'
                # 平仓信号：跌破20日最低价
                elif current_price < low_20:
                    signals[etf] = 'short'
                else:
                    signals[etf] = 'hold'
        
        return signals

    # 计算每日交易信号
    context.logger.info("开始计算每日交易信号...")
    
    all_dates = sorted(df_price['date'].unique())
    trade_signals = []
    
    for date in all_dates:
        # 检查是否处于休息状态
        if context.is_resting:
            context.rest_counter += 1
            if context.rest_counter >= context.rest_days:
                context.is_resting = False
                context.rest_counter = 0
            # 休息期间保持空仓
            trade_signals.append({
                'date': date,
                'instrument': None,
                'weight': 0.0
            })
            continue
            
        # 获取动量排名
        momentum_rank = get_rank(df_price, date, context.m_days)
        
        # 获取海龟策略信号
        turtle_signals = get_turtle_signals(df_price, date, context.window_size)
        
        # 结合两种策略
        selected_etfs = []
        
        # 优先选择动量排名前N且有海龟买入信号的ETF
        for etf in momentum_rank:
            if len(selected_etfs) >= context.target_num:
                break
            if turtle_signals.get(etf) == 'long':
                selected_etfs.append(etf)
        
        # 如果没有符合条件的ETF，选择动量排名前N的ETF
        if len(selected_etfs) == 0:
            selected_etfs = momentum_rank[:context.target_num]
        
        # 构建交易信号
        if len(selected_etfs) > 0:
            # 等权重分配
            weight = 1.0 / len(selected_etfs)
            for etf in selected_etfs:
                trade_signals.append({
                    'date': date,
                    'instrument': etf,
                    'weight': weight
                })
        else:
            # 空仓
            trade_signals.append({
                'date': date,
                'instrument': None,
                'weight': 0.0
            })
    
    # 处理为每日信号格式
    df_signals = pd.DataFrame(trade_signals)
    
    # 转换为每日持仓格式
    daily_signals = []
    current_date = None
    current_positions = {}
    
    for _, row in df_signals.iterrows():
        if row['date'] != current_date:
            if current_date is not None:
                # 保存前一日的持仓
                for etf, weight in current_positions.items():
                    daily_signals.append({
                        'date': current_date,
                        'instrument': etf,
                        'weight': weight
                    })
                # 添加空仓ETF
                for etf in context.etf_pool:
                    if etf not in current_positions:
                        daily_signals.append({
                            'date': current_date,
                            'instrument': etf,
                            'weight': 0.0
                        })
            
            current_date = row['date']
            current_positions = {}
            
            if pd.notna(row['instrument']):
                current_positions[row['instrument']] = row['weight']
        else:
            if pd.notna(row['instrument']):
                current_positions[row['instrument']] = row['weight']
    
    # 添加最后一日
    if current_date is not None:
        for etf, weight in current_positions.items():
            daily_signals.append({
                'date': current_date,
                'instrument': etf,
                'weight': weight
            })
        for etf in context.etf_pool:
            if etf not in current_positions:
                daily_signals.append({
                    'date': current_date,
                    'instrument': etf,
                    'weight': 0.0
                })
    
    df_signals = pd.DataFrame(daily_signals)
    
    context.logger.info(f"交易信号计算完成: {len(df_signals)} 个")
    
    # 每日调仓
    context.data = df_signals
    
    context.logger.info(f"策略数据准备完成，共{len(context.data)}条调仓信号")


def handle_data(context: bigtrader.IContext, data: bigtrader.IBarData):
    """处理交易信号"""
    current_date = data.current_dt.strftime('%Y-%m-%d')
    
    # 更新历史最高净值
    current_value = context.portfolio.portfolio_value
    if current_value > context.max_net_value:
        context.max_net_value = current_value
    
    # 检查是否触发回撤阈值
    if not context.is_resting:
        if context.max_net_value > 0:
            drawdown = (context.max_net_value - current_value) / context.max_net_value
            if drawdown > (1 - context.value_shrink):
                context.is_resting = True
                context.rest_counter = 0
                context.logger.info(f"触发回撤阈值 {drawdown:.2%}，开始休息 {context.rest_days} 天")
    
    # 如果处于休息状态，清仓
    if context.is_resting:
        context.rest_counter += 1
        if context.rest_counter >= context.rest_days:
            context.is_resting = False
            context.rest_counter = 0
            context.max_net_value = context.portfolio.portfolio_value
            context.logger.info(f"休息期满 {context.rest_days} 天，重置 is_resting=True")
        for symbol in context.portfolio.positions:
            position = context.portfolio.positions[symbol]
            if position.amount > 0:
                context.order_target(symbol, 0)
        return
    
    # 正常交易逻辑
    today_data = context.data[context.data['date'] == current_date]
    
    if len(today_data) == 0:
        return
    
    # 先处理需要卖出的持仓
    for symbol in context.portfolio.positions:
        position = context.portfolio.positions[symbol]
        if position.amount > 0:
            # 检查是否需要卖出
            target_row = today_data[today_data['instrument'] == symbol]
            if len(target_row) == 0 or target_row['weight'].iloc[0] == 0:
                # 需要卖出
                context.order_target(symbol, 0)
    
    # 再处理需要买入的标的
    for _, row in today_data.iterrows():
        if row['weight'] > 0:
            context.order_target_percent(row['instrument'], row['weight'])


# 运行策略回测
performance = bigtrader.run(
    market=bigtrader.Market.CN_FUND,
    frequency=bigtrader.Frequency.DAILY,
    start_date="2024-01-30",
    end_date="2025-09-08", 
    capital_base=3000000,
    initialize=initialize,
    handle_data=handle_data,
)

# 显示回测结果
performance.render()